# Password spraying: dataset synthétique avancé + pipeline

Génération de logs réalistes (normal + attaques) et calcul de features avancées par fenêtre de temps.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RNG = np.random.default_rng(42)
RAW = Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
PROC = Path('data/processed'); PROC.mkdir(parents=True, exist_ok=True)


## 1) Génération du dataset avancé

Colonnes: ts, user, src_ip, app, result, reason, user_agent, country

In [ ]:

from dataclasses import dataclass

@dataclass
class SynthConfig:
    start: str = '2026-02-01 08:00:00'
    minutes: int = 12*60
    n_users: int = 400
    n_normal_ips: int = 80
    apps: tuple = ('SSO','VPN','MAIL')

cfg = SynthConfig()


In [ ]:

def make_advanced_synth_logs(cfg: SynthConfig, rng=RNG):
    start = pd.Timestamp(cfg.start, tz='UTC')
    users = [f"user{idx:04d}" for idx in range(cfg.n_users)]
    normal_ips = [f"198.51.100.{i}" for i in range(1, cfg.n_normal_ips+1)]

    countries = ['FR','DE','ES','GB','US','NL']
    user_agents = ['Chrome','Firefox','Edge','MobileSafari','python-requests','curl']

    rows = []

    # Normal traffic
    for m in range(cfg.minutes):
        ts_base = start + pd.Timedelta(minutes=m)
        hour = ts_base.hour
        lam = 10 if 9 <= hour <= 18 else 4  # plus d'activité en journée
        n_events = rng.poisson(lam)

        for _ in range(n_events):
            ts = ts_base + pd.Timedelta(seconds=int(rng.integers(0, 60)))
            user = rng.choice(users)
            ip = rng.choice(normal_ips)
            app = rng.choice(cfg.apps, p=[0.65, 0.2, 0.15])
            ua = rng.choice(user_agents, p=[0.35,0.25,0.2,0.1,0.07,0.03])
            country = rng.choice(countries, p=[0.45,0.2,0.08,0.1,0.12,0.05])

            p_fail = {'SSO':0.06,'VPN':0.10,'MAIL':0.04}[app]
            fail = rng.random() < p_fail
            if fail:
                result = 'fail'
                reason = rng.choice(['bad_password','mfa_required','user_typo'], p=[0.75,0.05,0.20])
            else:
                result = 'success'
                reason = 'ok'

            rows.append((ts, user, ip, app, result, reason, ua, country))

    # Scenario A: password spray (1 IP -> beaucoup de users, surtout fail, rythme modéré)
    spray_ip = '203.0.113.50'
    spray_start = 3*60 + 20
    spray_dur = 45
    spray_users = rng.choice(users, size=140, replace=False)

    for m in range(spray_start, spray_start + spray_dur):
        ts_base = start + pd.Timedelta(minutes=m)
        n_attempts = rng.poisson(18)
        for _ in range(n_attempts):
            ts = ts_base + pd.Timedelta(seconds=int(rng.integers(0, 60)))
            user = rng.choice(spray_users)
            app = rng.choice(cfg.apps, p=[0.85, 0.1, 0.05])
            ua = 'python-requests'
            country = 'US'
            success = rng.random() < 0.012
            result = 'success' if success else 'fail'
            reason = 'ok' if success else 'bad_password'
            rows.append((ts, user, spray_ip, app, result, reason, ua, country))

    # Scenario B: brute force classique (1 user -> plein d'essais)
    brute_ip = '203.0.113.99'
    victim = rng.choice(users)
    brute_start = 6*60 + 10
    brute_dur = 20
    for m in range(brute_start, brute_start + brute_dur):
        ts_base = start + pd.Timedelta(minutes=m)
        n_attempts = rng.poisson(55)
        for _ in range(n_attempts):
            ts = ts_base + pd.Timedelta(seconds=int(rng.integers(0, 60)))
            app = rng.choice(cfg.apps, p=[0.7, 0.2, 0.1])
            ua = rng.choice(['curl','python-requests'], p=[0.4,0.6])
            country = 'NL'
            rows.append((ts, victim, brute_ip, app, 'fail', 'bad_password', ua, country))

    # Scenario C: misconfig / outage (beaucoup de fail, mais dispersé sur IPs normales)
    outage_start = 9*60
    outage_dur = 25
    affected_users = rng.choice(users, size=60, replace=False)
    for m in range(outage_start, outage_start + outage_dur):
        ts_base = start + pd.Timedelta(minutes=m)
        n_events = rng.poisson(35)
        for _ in range(n_events):
            ts = ts_base + pd.Timedelta(seconds=int(rng.integers(0, 60)))
            user = rng.choice(affected_users)
            ip = rng.choice(normal_ips)
            app = 'VPN'
            ua = rng.choice(['Chrome','Edge','MobileSafari'], p=[0.5,0.3,0.2])
            country = rng.choice(['FR','DE','GB'], p=[0.6,0.25,0.15])
            rows.append((ts, user, ip, app, 'fail', 'mfa_required', ua, country))

    df = pd.DataFrame(rows, columns=['ts','user','src_ip','app','result','reason','user_agent','country'])
    df = df.sort_values('ts').reset_index(drop=True)
    return df

logs = make_advanced_synth_logs(cfg)
logs.to_csv(RAW/'auth_logs_advanced.csv', index=False)
logs.head(), len(logs)


## 2) Features avancées par fenêtre (10 min) et par IP

In [ ]:

# Chargement
logs = pd.read_csv(RAW/'auth_logs_advanced.csv')
logs['ts'] = pd.to_datetime(logs['ts'], utc=True)
logs = logs.sort_values('ts')

# Flags
logs['is_fail'] = (logs['result'] == 'fail').astype(int)
logs['is_success'] = (logs['result'] == 'success').astype(int)

WINDOW = pd.Timedelta('10min')
STEP = '1min'

# On travaille sur un index temps
logs = logs.set_index('ts')

# Agrégations de base par IP sur des pas d'1 minute
base = (
    logs.groupby('src_ip')
        .resample(STEP)
        .agg(
            n_attempts=('result','size'),
            n_fail=('is_fail','sum'),
            n_success=('is_success','sum'),
            n_users=('user','nunique'),
            n_apps=('app','nunique'),
            n_user_agents=('user_agent','nunique'),
        )
)

# Rolling sur 10 minutes (somme sur les compteurs)
roll = (
    base.groupby(level=0)
        .rolling(WINDOW)
        .sum()
)

feat = roll.reset_index().rename(columns={'ts':'window_end'})
feat['fail_rate'] = feat['n_fail'] / feat['n_attempts'].replace(0, np.nan)
feat['attempts_per_min'] = feat['n_attempts'] / (WINDOW.total_seconds()/60)

# triage simple
feat['success_after_fail'] = ((feat['n_success'] > 0) & (feat['n_fail'] >= 20)).astype(int)

feat.to_csv(PROC/'features_ip_windows.csv', index=False)
feat.head()


## 3) Modèle (IsolationForest) + export alertes

In [ ]:

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

cols = ['n_attempts','n_fail','n_success','n_users','n_apps','n_user_agents','fail_rate','attempts_per_min','success_after_fail']
X = feat[cols].fillna(0.0)

scaler = StandardScaler()
Xn = scaler.fit_transform(X)

model = IsolationForest(n_estimators=300, random_state=42, contamination=0.01)
model.fit(Xn)

feat['anomaly_score'] = -model.score_samples(Xn)
feat['is_anomaly'] = (model.predict(Xn) == -1).astype(int)

alerts = feat[feat['is_anomaly']==1].sort_values('anomaly_score', ascending=False)
alerts.to_csv(PROC/'alerts.csv', index=False)

alerts.head(20)
